# Advanced usage: The `MorphingWorkflow` classes: `MorphingWorkflowGlobal`, `MorphingWorkflowEurope` and `MorphingWorkflowCordex`

This notebook demonstrates the advanced, step-by-step approach to morphing EPW files using the specialized `MorphingWorkflow` classes (`MorphingWorkflowGlobal`, `MorphingWorkflowEurope` and `MorphingWorkflowCordex`).

While the `morph_epw_*` functions are great for direct, one-shot tasks, the workflow classes are designed for complex projects where you need full control over **filename parsing, custom renaming, and process validation**. They enforce a safe, multi-step process that allows you to review and confirm each stage before executing the time-consuming morphing computation.

This notebook will focus on **`MorphingWorkflowGlobal`**, and will show quick examples using **`MorphingWorkflowEurope`** and **`MorphingWorkflowCordex`** highlighting the differences.

**Supported binaries:** Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope.

**New in pyfwg v0.4.0:** `pyfwg` also supports the **FWG Engine**, the shared engine of **Global v5.x**, **Europe v3.x** and the **CORDEX regional** distributions. `MorphingWorkflowGlobal` and `MorphingWorkflowEurope` work with Global v5.x and Europe v3.x, and the new `MorphingWorkflowCordex` class handles the CORDEX regional distributions.


## Configuration

The supported examples use Global 5.2.0, Europe EUR-11 3.2.0 and the other eight CORDEX-CMIP5 distributions 1.2.0. Older Global/Europe releases are outside the supported scope. Download JARs from the [FWG website](https://future-weather-generator.adai.pt/).

Paths are read from environment variables first, then an optional `fwg.local.json` in the project/working folder, then the defaults below. This local JSON file is excluded from Git. JARs can be on an external drive; keep it connected throughout a run. A path containing spaces needs no special escaping in a Python string.

- `FWG_JAR_DIR`: folder of regional JARs (default: project root); keep one version per domain.
- `FWG_GLOBAL_JAR_PATH`: optional full path to Global 5.2.0 if it is in another folder.
- `FWG_EPW_DIR`: real regional baselines named `<domain>.epw`. Global examples use `EUR-11.epw`. Retained validation samples are used if present; otherwise supply `epws/cordex`. Public sample links are in the regional notebook. Do not alter station coordinates to fit a region.
- `FWG_CACHE_DIR`: extracted climate data (default: `~/fwg_model_cache`). EUR-11's full ensemble alone contains about 11.3 GB before extraction overhead. Cache location is independent of the JAR folder.
- `FWG_OUTPUT_DIR`: results, Excel workbooks and diagnostics (default: `build/cordex-tutorial`).
- `FWG_JAVA`: optional Java executable; otherwise Java 17+ must be on PATH.

`RUN_MORPHING = False` prepares every example without starting Java. Set it to `True` to generate weather. Notebook outputs are cleared before distribution; generated files are kept for inspection.


In [ ]:
import json
import os
from pathlib import Path
from pyfwg import enable_console_logging

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyfwg" / "__init__.py").is_file()),
    Path.cwd(),
)
local_file = PROJECT_ROOT / "fwg.local.json"
local_paths = json.loads(local_file.read_text(encoding="utf-8")) if local_file.is_file() else {}

def configured_path(name, default):
    return Path(os.environ.get(name, local_paths.get(name, str(default)))).expanduser().resolve()

validation_epws = PROJECT_ROOT / "build/cordex-validation/epws"
default_epws = validation_epws if validation_epws.is_dir() else PROJECT_ROOT / "epws/cordex"
FWG_JAR_DIR = configured_path("FWG_JAR_DIR", PROJECT_ROOT)
FWG_GLOBAL_JAR_PATH = configured_path("FWG_GLOBAL_JAR_PATH", FWG_JAR_DIR / "FutureWeatherGenerator-CMIP6-Global-v5.2.0.jar")
FWG_EPW_DIR = configured_path("FWG_EPW_DIR", default_epws)
FWG_CACHE_DIR = configured_path("FWG_CACHE_DIR", Path.home() / "fwg_model_cache")
FWG_OUTPUT_DIR = configured_path("FWG_OUTPUT_DIR", PROJECT_ROOT / "build/cordex-tutorial")
java_executable = os.environ.get("FWG_JAVA", local_paths.get("FWG_JAVA"))
if java_executable:
    java_path = Path(java_executable).expanduser().resolve()
    if not java_path.is_file():
        raise FileNotFoundError(f"Java executable not found: {java_path}")
    os.environ["PATH"] = str(java_path.parent) + os.pathsep + os.environ.get("PATH", "")
RUN_MORPHING = False
enable_console_logging()
print("Regional JARs:", FWG_JAR_DIR)


### Read a regional JAR and select compatible inputs

This helper reads the actual manifest, including renamed JARs. It selects one model by default, or every ensemble-eligible model with `ensemble=True`, then finds a scenario and period shared by the selection. Empty selections and ambiguous JAR folders fail with an explicit message. The returned dictionary uses the public `pyfwg` parameters; inspect or edit it before running an example.

Caches are separated by distribution ID **and version**, since the same GCM–RCM name can occur in different regions. All examples use `IDW`. `BI`/`1` needs a rectilinear geographic grid and is rejected for these rotated/projected regional grids. One distinct model disables ensemble mode; aliases containing underscores and hyphens refer to the same model. CORDEX Engine distributions do not implement UHI.


In [ ]:
from pyfwg import get_fwg_distribution_info

def regional_jar(domain):
    matches = []
    for jar in sorted(FWG_JAR_DIR.glob("*.jar")):
        info = get_fwg_distribution_info(str(jar))
        if info and info["framework"] == "CORDEX" and info["domain"] == domain:
            matches.append((jar, info))
    if len(matches) != 1:
        raise ValueError(f"Expected one {domain} JAR in {FWG_JAR_DIR}; found {len(matches)}. Check the folder and versions.")
    return matches[0]

def regional_inputs(domain, ensemble=False):
    jar, info = regional_jar(domain)
    models = info["ensemble_models"] if ensemble else info["models"][:1]
    if not models or (ensemble and len(models) < 2):
        raise ValueError(f"{domain}: need one model, or at least two eligible models for an ensemble.")
    details = info["model_details"]
    scenarios = [s for s in info["scenarios"] if all(s in details[m]["scenarios"] for m in models)]
    periods = [t for t in info["timeframes"] if all(t in details[m]["timeframes"] for m in models)]
    if not scenarios or not periods:
        raise ValueError(f"{domain}: selected models have no shared scenario or period.")
    epw = FWG_EPW_DIR / f"{domain}.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply a real baseline inside {domain} at {epw}, or set FWG_EPW_DIR.")
    options = dict(
        epw_paths=str(epw), fwg_jar_path=str(jar),
        fwg_rcm_pairs=list(models), fwg_create_ensemble=ensemble,
        fwg_scenarios=[scenarios[0]], fwg_timeframes=[periods[0]],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )
    return info, options


In [ ]:
def global_inputs():
    info = get_fwg_distribution_info(str(FWG_GLOBAL_JAR_PATH))
    if not info or info["framework"] != "CMIP" or info.get("domain"):
        raise ValueError(f"Supply a Global FWG Engine JAR at {FWG_GLOBAL_JAR_PATH}")
    epw = FWG_EPW_DIR / "EUR-11.epw"
    if not epw.is_file():
        raise FileNotFoundError(f"Supply the Global example baseline at {epw}")
    model = "CanESM5" if "CanESM5" in info["models"] else info["models"][0]
    details = info["model_details"][model]
    scenario = next(s for s in info["scenarios"] if s in details["scenarios"])
    period = next(t for t in info["timeframes"] if t in details["timeframes"])
    return info, dict(
        epw_paths=str(epw), fwg_jar_path=str(FWG_GLOBAL_JAR_PATH),
        fwg_gcms=[model], fwg_create_ensemble=False, fwg_add_uhi=False,
        fwg_scenarios=[scenario], fwg_timeframes=[int(period)],
        fwg_interpolation_method_id="IDW", fwg_output_profile="MINIMAL",
        fwg_use_multithreading=False,
        fwg_model_unzipped_folder=str(FWG_CACHE_DIR / info["distribution_id"] / info["version"]),
        fwg_extra_args={"threads": 1, "io_threads": 1, "save_technical_log": True},
        delete_temp_files=False,
    )


## The Three-Step Workflow

The class is designed to be used as a state machine, guiding you through a logical sequence:

1.  **`map_categories()`**: Analyze the input filenames to extract meaningful data (like city, building type, etc.).
2.  **`configure_and_preview()`**: Define all execution parameters, validate them, and generate a "dry run" plan of how the final files will be named and organized.
3.  **`execute_morphing()`**: Run the final computation, confident that your plan and configuration are correct.


## Global 5.2.0: map, preview, execute

Map a `station` category from the input name. For your own files, `input_filename_pattern` can extract named regex groups such as `(?P<city>.+)_(?P<station>.+)`; `keyword_mapping` normalizes those values. Global filenames require `{ssp}` and `{year}`. Regional workflows use `{rcp}` and `{year}`.


In [ ]:
from pyfwg import MorphingWorkflowGlobal
info_global, global_options = global_inputs()
workflow_global = MorphingWorkflowGlobal()
workflow_global.map_categories(
    epw_files=[global_options["epw_paths"]],
    keyword_mapping={"station": {"Seville": "EUR-11"}},
)
workflow_global.configure_and_preview(
    **{k: v for k, v in global_options.items() if k != "epw_paths"},
    final_output_dir=str(FWG_OUTPUT_DIR / "global-workflow"),
    temp_base_dir=str(FWG_OUTPUT_DIR / "temp-global-workflow"),
    output_filename_pattern="{station}_{ssp}_{year}",
)
if not workflow_global.is_config_valid:
    raise ValueError("Review the Global configuration warnings.")
if RUN_MORPHING:
    workflow_global.execute_morphing()


### Europe v3.x example (FWG Engine)

Europe EUR-11 v3.2.0 has passed real single-model and full seven-member ensemble generation checks. `MorphingWorkflowEurope` and `MorphingWorkflowCordex` both accept it. Run Configuration and the regional helper first. This example uses `EUR-11.epw`, one model and a shared scenario/period from the actual catalog.

Use `ensemble=True` to select the complete ensemble, and allow about 11.3 GB for extracted climate data plus temporary overhead. Setting `RUN_MORPHING=True` executes the prepared workflow.


In [ ]:
from pyfwg import MorphingWorkflowEurope

info_europe, europe_options = regional_inputs("EUR-11", ensemble=False)
workflow_eu_v3 = MorphingWorkflowEurope()
workflow_eu_v3.map_categories(
    epw_files=[europe_options["epw_paths"]],
    keyword_mapping={"region": {"EUR-11": "EUR-11"}},
)
workflow_eu_v3.configure_and_preview(
    **{key: value for key, value in europe_options.items() if key != "epw_paths"},
    final_output_dir=str(FWG_OUTPUT_DIR / "europe-workflow"),
    temp_base_dir=str(FWG_OUTPUT_DIR / "temp-europe-workflow"),
    output_filename_pattern="{region}_{rcp}_{year}",
)
if not workflow_eu_v3.is_config_valid:
    raise ValueError("Review the Europe configuration warnings before running.")
if RUN_MORPHING:
    workflow_eu_v3.execute_morphing()


The preview uses `{rcp}` and `{year}`. Output, temporary diagnostics and climate cache are kept in separate directories.


In [ ]:
print("Europe workflow output:", FWG_OUTPUT_DIR / "europe-workflow")


In [ ]:
print("Europe diagnostics:", FWG_OUTPUT_DIR / "temp-europe-workflow")


## Using `MorphingWorkflowCordex`

The class supports EUR-11, MNA-22, AFR-22, NAM-44, CAM-44, SAM-44, WAS-22, SEA-22 and AUS-22. Its model parameter is `fwg_rcm_pairs`, and output patterns use `{rcp}`. Metadata from the selected JAR controls the regional catalog. There are no UHI/LCZ parameters.

The example below uses an actual Cairo baseline named `MNA-22.epw`, selects the complete two-model MNA-22 ensemble, and preserves diagnostics. Choose a different `DOMAIN` and matching EPW for another region; Java verifies native model coverage. Cache directories include the distribution ID and version. See the [regional notebook](cordex_regions_example.ipynb) for the full nine-region catalog and a batch example.


In [ ]:
from pyfwg import MorphingWorkflowCordex

DOMAIN = "MNA-22"
info_cordex, cordex_options = regional_inputs(DOMAIN, ensemble=True)
workflow_cordex = MorphingWorkflowCordex()
workflow_cordex.map_categories(
    epw_files=[cordex_options["epw_paths"]],
    keyword_mapping={"region": {DOMAIN: DOMAIN}},
)
workflow_cordex.configure_and_preview(
    **{key: value for key, value in cordex_options.items() if key != "epw_paths"},
    final_output_dir=str(FWG_OUTPUT_DIR / "workflow" / DOMAIN),
    temp_base_dir=str(FWG_OUTPUT_DIR / "temp-workflow" / DOMAIN),
    output_filename_pattern="{region}_{rcp}_{year}",
)
if not workflow_cordex.is_config_valid:
    raise ValueError("Review the regional configuration warnings before running.")


In [ ]:
print("Detected domain:", workflow_cordex.cordex_domain)
if RUN_MORPHING:
    workflow_cordex.execute_morphing()
else:
    print("Preview only; set RUN_MORPHING=True to generate weather.")


### Inspect results, SolarQC and disk usage

For EPW output, each selected scenario/period normally produces a future EPW and its STAT. The API returns their paths; workflows move and rename them into `final_output_dir`. Check that files exist: a failed Java run is logged and may leave an empty result list. A preview alone does not establish that the EPW passes the engine's checks.

These examples keep `delete_temp_files=False` and technical logs. For SolarQC rejection, inspect `<temp_base_dir>/<EPW stem>/metadata/solarqc_preprocessing_failure.txt` and the `logs/application` folder. During the 2026-10-28 integration checks, New Delhi and Suvarnabhumi samples were rejected; unmodified public Mumbai and Singapore samples passed. Changing the JAR or station coordinates is not a repair for an unsuitable baseline. `MINIMAL` reduces auxiliary output; mandatory diagnostics can still be written.

After reviewing results and diagnostics, you can remove the corresponding temporary directory and the **specific distribution/version cache** when no run is using it. `delete_temp_files=True` removes per-EPW temporary files after a successful run; it does not clear a separately configured shared cache. No cleanup cell in these regional examples removes your results or caches automatically. An iterator reuses caches but does not evict previous regions: with limited space, run one region at a time and clean its cache before continuing.

The downloaded TMYx test samples cover 2009–2023, whereas the CORDEX climate reference period is 1976–2005. FWG reports this mismatch. These are software integration examples; choose and assess the baseline/reference period for your study. See the [CORDEX regional notebook](cordex_regions_example.ipynb) for the nine-region validation matrix and source links.


In [ ]:
print("Regional workflow output:", FWG_OUTPUT_DIR / "workflow" / DOMAIN)


In [ ]:
print("Regional diagnostics:", FWG_OUTPUT_DIR / "temp-workflow" / DOMAIN)
